# QM9 HOMO–LUMO gap: DimeNet++ vs QDimeNet++

This notebook reproduces the inference behind the thesis's QM9 chapter. It runs the classical **DimeNet++** baseline and **QDimeNet++**, which is the same network plus a QFIM edge-rescale adapter. Both models were trained on the HOMO–LUMO gap (run006, seed 42). On the full test set of 52,313 molecules, the seed-42 models reach a test MAE of **71.45 meV** (DimeNet++) and **67.99 meV** (QDimeNet++).

Here both models run on a demo set: the first 5,000 molecules of the same test file. Expect the demo MAEs to differ from the full-test values by a few meV. The demo file also holds the thesis's own saved test predictions for these molecules, and the last cell checks that the predictions made here match them.

**Model.** Both models are DimeNet++ on heavy atoms only (hidden 128, 4 interaction blocks, cutoff 5 Å). QDimeNet++ handles each directed edge j → i of the molecular graph as follows:

1. It takes the 6×6 block Q[j, i] of the two atoms' qubits from the QFIM of the bioQINN circuit (10 qubits, 2 layers of Rot(φ, θ, ω) per qubit).
2. A small CNN plus an MLP encodes the block into a score s ∈ (−1, 1).
3. The edge state is rescaled, m_ji ← m_ji · (1 + α·s), after the embedding block and after each of the 4 interaction blocks.

The encoder, the score MLP and the scalar gate α (initialised at 0) are shared by all five injection points. Together they add 5,018 parameters. In the trained seed-42 model the score is saturated at s = −1 on every demo edge. The adapter therefore acts as a constant edge damping by 1 − α ≈ 0.22, which the fourth cell prints.

**Requirements:** `torch`, `torch_geometric` and `numpy`. `sympy` is also needed for the DimeNet++ basis functions, and it comes with `torch`. The notebook does not need `torch_cluster` or `torch_sparse`. It was tested with torch 2.9.1, torch_geometric 2.7.0 and numpy 1.26 on CPU. With 4 CPU threads the whole notebook runs in about a minute.

**Files:**
- `thesis_models/qdimenet.py`: the models.
- `weights/qm9/*.pt`: the thesis checkpoints, unchanged.
- `data/qm9_test_demo.npz`: the demo molecules, built by `tools/export_qm9.py`, which also documents where every array comes from.

In [ ]:
import time

import numpy as np
import torch
import torch_geometric

from thesis_models.qdimenet import (DimeNetPP, QDimeNetPP, count_parameters, load_demo,
                                    molecule_batches, radius_graph)

DATA = "data/qm9_test_demo.npz"
WEIGHTS = {"DimeNet++": "weights/qm9/dimenet_pp_run006_seed42.pt",
           "QDimeNet++": "weights/qm9/qdimenet_pp_run006_seed42.pt"}
FULL_TEST_MAE_MEV = {"DimeNet++": 71.45, "QDimeNet++": 67.99}      # thesis, seed 42, 52,313 test molecules
EXPECTED_PARAMS = {"DimeNet++": 1_885_830, "QDimeNet++": 1_890_848}
BATCH_SIZE = 128                                                    # as in the thesis evaluation

torch.set_grad_enabled(False)
print(f"torch {torch.__version__} | torch_geometric {torch_geometric.__version__} | numpy {np.__version__} | "
      f"{torch.get_num_threads()} CPU threads")

In [ ]:
demo = load_demo(DATA)
for key in ["z", "pos", "atom_ptr", "qfim", "gap_eV", "test_index"]:
    print(f"{key:10s} {str(demo[key].shape):22s} {demo[key].dtype}")

n_mol = demo["gap_eV"].size
atoms_per_mol = np.diff(demo["atom_ptr"])
gap = demo["gap_eV"].astype(np.float64)
mu, sigma = float(demo["target_mean_eV"]), float(demo["target_std_eV"])
print(f"\n{n_mol} test molecules (rows {demo['test_index'][0]}-{demo['test_index'][-1]} of the test file), "
      f"{demo['z'].size} heavy atoms ({atoms_per_mol.min()}-{atoms_per_mol.max()} per molecule)")
print("QFIM per molecule:", demo["qfim"].shape[1:], "= (qubit a, qubit b, Rot parameter of a, Rot parameter of b);",
      "qubit a = atom row a (heavy atoms first, then hydrogens)")
print(f"gap: mean {gap.mean():.3f} eV, std {gap.std():.3f} eV | "
      f"training standardisation mu = {mu:.4f} eV, sigma = {sigma:.4f} eV")

In [ ]:
models = {"DimeNet++": DimeNetPP(), "QDimeNet++": QDimeNetPP()}
for name, model in models.items():
    model.load_state_dict(torch.load(WEIGHTS[name], map_location="cpu", weights_only=True), strict=True)
    model.eval()
    n_params = count_parameters(model)
    assert n_params == EXPECTED_PARAMS[name], n_params
    print(f"{name:11s} {n_params:>9,} parameters")

qdn = models["QDimeNet++"]
n_head, n_score = count_parameters(qdn.qfim_head), count_parameters(qdn.qfim_scale)
print(f"QFIM adapter: {n_head + n_score + 1:,} parameters "
      f"(CNN encoder {n_head:,}, score MLP {n_score}, gate alpha 1)")

alpha = qdn.qfim_residual_gate.item()
print(f"\nlearned gate alpha = {alpha:.4f} (beta = {qdn.qfim_rescale_beta:g})")
print(f"edge multiplier 1 + alpha*s is bounded by ({1 - abs(alpha):.4f}, {1 + abs(alpha):.4f}); "
      f"edge damping at s = -sign(alpha): 1 - |alpha| = {1 - abs(alpha):.4f}")

# The score s depends only on the QFIM block of each edge; look at what it does on the demo molecules.
s = torch.cat([qdn.edge_score(qfim, radius_graph(pos, batch, 5.0), batch).view(-1)
               for _, z, pos, batch, qfim in molecule_batches(demo, 512)]).double()
mult = 1 + alpha * s
print(f"on the {s.numel():,} demo edges: s in [{s.min():.6f}, {s.max():.6f}] (max 1 + s = {(1 + s).max():.1e}); "
      f"multiplier 1 + alpha*s: mean {mult.mean():.6f}, std {mult.std():.1e}")

In [ ]:
def predict(model, use_qfim):
    pred = np.empty(n_mol)
    for mols, z, pos, batch, qfim in molecule_batches(demo, BATCH_SIZE):
        out = model(z, pos, batch, qfim) if use_qfim else model(z, pos, batch)
        pred[mols] = out.double().numpy() * sigma + mu               # standardised -> eV
    return pred

reference = {"DimeNet++": demo["ref_pred_dimenet_pp"], "QDimeNet++": demo["ref_pred_qdimenet_pp"]}
preds = {}
print(f"{'model':11s} | {'demo MAE':>9s} | {'thesis full-test MAE':>20s} | {'max |pred - thesis pred|':>24s} | time")
for name, model in models.items():
    t0 = time.time()
    preds[name] = predict(model, use_qfim=(name == "QDimeNet++"))
    mae = 1e3 * np.mean(np.abs(preds[name] - gap))
    max_dev = np.max(np.abs(preds[name] - reference[name]))
    print(f"{name:11s} | {mae:5.2f} meV | {FULL_TEST_MAE_MEV[name]:16.2f} meV | {1e3 * max_dev:20.4f} meV | "
          f"{time.time() - t0:.0f} s")
    assert max_dev < 1e-3, "predictions do not match the thesis test predictions"

diff = 1e3 * (np.abs(preds["DimeNet++"] - gap) - np.abs(preds["QDimeNet++"] - gap))
print(f"\nQDimeNet++ - DimeNet++ on the demo set: {-diff.mean():+.2f} meV MAE "
      f"(paired standard error {diff.std(ddof=1) / np.sqrt(n_mol):.2f} meV)")